In [ ]:
import subprocess
import time
import os
import json
import datetime
import tempfile
import random
import shutil
import asyncio
import signal
import glob
import re
import threading
import importlib.util
from urllib.parse import urlparse, parse_qs


# ==============================================================================
# 0. ONE-TIME INSTALL  (parallel + timed, to keep Kaggle start-up short)
#    Also fetches NVIDIA's *graphics* libraries so the GPU can render WebGL.
#    (Kaggle's GPU only ships compute libraries; that is why 3D was CPU-only.)
# ==============================================================================

_T0 = time.time()


def _log(msg):
    print(f"[setup {time.time() - _T0:4.0f}s] {msg}")


def _sh(cmd):
    """Run a shell command quietly; returns the CompletedProcess."""
    return subprocess.run(
        cmd, shell=True, executable="/bin/bash",
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
    )


def _out(cmd):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash",
                       capture_output=True, text=True)
    return r.returncode, (r.stdout or "").strip()


def is_installed(cmd):
    return shutil.which(cmd) is not None


def _pkg_installed(pkg):
    return subprocess.run(f"dpkg -s {pkg}", shell=True,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0


def _has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


# ---- NVIDIA detection -------------------------------------------------------
HAS_NVIDIA = is_installed("nvidia-smi") and _out("nvidia-smi -L")[0] == 0
NV_VER = None        # e.g. "535.129.03"
NV_NAME = None       # e.g. "Tesla T4"
NV_BUSID = None      # e.g. "00000000:00:04.0"
if HAS_NVIDIA:
    _rc, _o = _out("nvidia-smi --query-gpu=driver_version,name,pci.bus_id --format=csv,noheader")
    if _rc == 0 and _o:
        _parts = [x.strip() for x in _o.splitlines()[0].split(",")]
        if len(_parts) >= 3:
            NV_VER, NV_NAME, NV_BUSID = _parts[0], _parts[1], _parts[2]

NV_DIR = f"/tmp/NVIDIA-Linux-x86_64-{NV_VER}" if NV_VER else None   # extracted installer
NV_LIBS = "/tmp/nvlibs"                                            # curated libs + SONAME links
NV_OK = False                                                      # True when graphics libs are ready

# command -> apt package that provides it
_CMD_TO_PKG = {
    "Xvfb": "xvfb",
    "ffmpeg": "ffmpeg",
    "pulseaudio": "pulseaudio",
    "pactl": "pulseaudio-utils",
    "matchbox-window-manager": "matchbox-window-manager",
    "wmctrl": "wmctrl",
    "readelf": "binutils",
    "curl": "curl",
}
_missing_pkgs = sorted({pkg for cmd, pkg in _CMD_TO_PKG.items() if not is_installed(cmd)})
if _missing_pkgs:
    _missing_pkgs += ["dbus-x11", "xdotool"]
if HAS_NVIDIA:
    for _p in ["libvulkan1", "libgl1", "libegl1", "libglx0", "xserver-xorg-core"]:
        if not _pkg_installed(_p):
            _missing_pkgs.append(_p)
_missing_pkgs = sorted(set(_missing_pkgs))

_state = {"browser_installed_now": False}


def _install_system():
    if not _missing_pkgs:
        _log("System packages already present - skipping apt.")
        return
    _log(f"apt: installing {', '.join(_missing_pkgs)} ...")
    pk = " ".join(_missing_pkgs)
    base = "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
    r = _sh(f"apt-get update -y -qq && {base}--no-install-recommends {pk}")
    if r.returncode != 0:
        _log("Lean apt install failed - retrying with full dependencies...")
        r = _sh(f"apt-get update -y -qq && {base}{pk}")
        if r.returncode != 0:
            print((r.stderr or "")[-1500:])
            raise RuntimeError("apt-get install failed.")
    _log("apt: done.")


def _install_python():
    pip_pkgs = []
    if not _has_module("playwright"):
        pip_pkgs.append("playwright")
    if not _has_module("requests"):
        pip_pkgs.append("requests")
    if not _has_module("httpx"):
        pip_pkgs.append("httpx")
    if not (_has_module("googleapiclient") and _has_module("google.oauth2")):
        pip_pkgs += ["google-api-python-client", "google-auth", "google-auth-oauthlib"]
    if not _has_module("nepali_datetime"):
        pip_pkgs.append("nepali-datetime")

    if pip_pkgs:
        _log(f"pip: installing {', '.join(pip_pkgs)} ...")
        r = _sh("pip install -q " + " ".join(pip_pkgs))
        if r.returncode != 0:
            print((r.stderr or "")[-1500:])
            raise RuntimeError("pip install failed.")
        _log("pip: done.")
    else:
        _log("Python packages already present - skipping pip.")

    chk = subprocess.run(
        "playwright install --dry-run chromium 2>&1 | "
        "grep -qi 'is already downloaded' && echo YES || echo NO",
        shell=True, capture_output=True, text=True, executable="/bin/bash"
    )
    if "YES" not in chk.stdout:
        _log("playwright: downloading Chromium ...")
        r = _sh("playwright install chromium")
        if r.returncode != 0:
            print((r.stderr or "")[-1500:])
            raise RuntimeError("playwright install chromium failed.")
        _state["browser_installed_now"] = True
        _log("playwright: Chromium downloaded.")
    else:
        _log("Chromium already present - skipping download.")


def _install_nvidia_userspace():
    """Download NVIDIA's installer matching the running kernel driver and extract
    it (nothing is installed into the system). It contains the OpenGL / Vulkan /
    EGL / NVENC libraries that Kaggle does not mount into the container."""
    if glob.glob(f"{NV_DIR}/libGLX_nvidia.so.*"):
        _log("nvidia: graphics libraries already extracted.")
        return
    run = f"/tmp/NVIDIA-Linux-x86_64-{NV_VER}.run"
    urls = [
        f"https://us.download.nvidia.com/tesla/{NV_VER}/NVIDIA-Linux-x86_64-{NV_VER}.run",
        f"https://us.download.nvidia.com/XFree86/Linux-x86_64/{NV_VER}/NVIDIA-Linux-x86_64-{NV_VER}.run",
    ]
    ok = False
    for u in urls:
        _log(f"nvidia: downloading driver package {NV_VER} (about 300 MB) ...")
        _sh(f"rm -f {run}")
        r = _sh(f"curl -fL --retry 3 --connect-timeout 20 -o {run} {u}")
        if r.returncode == 0 and os.path.exists(run) and os.path.getsize(run) > 50 * 1024 * 1024:
            ok = True
            break
    if not ok:
        raise RuntimeError(f"could not download NVIDIA package {NV_VER} (is Internet ON in Kaggle?)")
    _log("nvidia: extracting ...")
    r = _sh(f"cd /tmp && sh {run} -x")
    _sh(f"rm -f {run}")
    if not glob.glob(f"{NV_DIR}/libGLX_nvidia.so.*"):
        print((r.stderr or "")[-800:])
        raise RuntimeError("NVIDIA package extracted but graphics libraries were not found.")
    _log("nvidia: done.")


def _install_nvidia_safe():
    try:
        _install_nvidia_userspace()
    except Exception as e:
        print(f"[setup] ⚠️ NVIDIA graphics libraries unavailable: {e}")


_SKIP_LIBS = re.compile(
    r"^(libGL|libEGL|libGLX|libOpenGL|libGLdispatch|libGLESv1_CM|libGLESv2|"
    r"libcuda|libnvidia-ml|libnvidia-ptxjitcompiler)\.so"
)


def _link_nvidia_libs():
    """Create a clean folder with the NVIDIA libs + the SONAME links they need.
    Generic GL libs (libGL, libEGL ...) are NOT included so the system ones stay."""
    global NV_OK
    if not NV_DIR or not glob.glob(f"{NV_DIR}/libGLX_nvidia.so.*"):
        return
    os.makedirs(NV_LIBS, exist_ok=True)
    for f in glob.glob(f"{NV_DIR}/*.so*"):
        base = os.path.basename(f)
        if _SKIP_LIBS.match(base):
            continue
        dst = os.path.join(NV_LIBS, base)
        if not os.path.lexists(dst):
            os.symlink(f, dst)
        r = subprocess.run(["readelf", "-d", f], capture_output=True, text=True)
        m = re.search(r"Library soname: \[(.+?)\]", r.stdout or "")
        if m:
            sod = os.path.join(NV_LIBS, m.group(1))
            if not os.path.lexists(sod):
                os.symlink(f, sod)

    glx = os.path.join(NV_LIBS, "libGLX_nvidia.so.0")
    egl = os.path.join(NV_LIBS, "libEGL_nvidia.so.0")
    if not os.path.exists(glx):
        return
    with open(os.path.join(NV_LIBS, "nvidia_icd.json"), "w") as fh:
        json.dump({"file_format_version": "1.0.0",
                   "ICD": {"library_path": glx, "api_version": "1.3.0"}}, fh)
    with open(os.path.join(NV_LIBS, "10_nvidia.json"), "w") as fh:
        json.dump({"file_format_version": "1.0.0",
                   "ICD": {"library_path": egl}}, fh)
    NV_OK = True


_errors = []


def _guard(fn):
    def runner():
        try:
            fn()
        except Exception as e:
            _errors.append(f"{fn.__name__}: {e}")
    return runner


# apt, pip/Chromium and the NVIDIA download use different resources: run together
_threads = [
    threading.Thread(target=_guard(_install_system)),
    threading.Thread(target=_guard(_install_python)),
]
if HAS_NVIDIA and NV_VER:
    _threads.append(threading.Thread(target=_install_nvidia_safe))
for _t in _threads:
    _t.start()
for _t in _threads:
    _t.join()

if _errors:
    raise RuntimeError("Setup failed: " + " | ".join(_errors))

# Chromium's system libraries (needs apt, so it runs after the parallel part)
if _state["browser_installed_now"]:
    _log("playwright: installing Chromium system libraries ...")
    _r = _sh("playwright install-deps chromium")
    if _r.returncode != 0:
        print((_r.stderr or "")[-1500:])
    _log("playwright: system libraries done.")

if HAS_NVIDIA and NV_VER:
    _link_nvidia_libs()
    _log(f"GPU: {NV_NAME} | driver {NV_VER} | graphics libs ready: {NV_OK}")
elif HAS_NVIDIA:
    _log("GPU found but the driver version could not be read - GPU rendering disabled.")
else:
    _log("No NVIDIA GPU visible (is the Kaggle accelerator set to GPU?).")

_log("Setup finished.")

import httpx
import nepali_datetime
import requests
from playwright.async_api import async_playwright


# ==============================================================================
# 1. PROCESS VARIABLES
# ==============================================================================

ffmpeg_proc = None
wm_proc = None
display_proc = None       # Xvfb or Xorg(NVIDIA)
pulseaudio_proc = None

live_video_id = None      # Facebook live id (set when live is created)
flag_was_set = False      # True once we have written the "playing" flag = True

yt_service = None         # YouTube API client (set when YouTube is used)
yt_broadcast_id = None    # YouTube broadcast id (set when broadcast is created)


# ==============================================================================
# 2. CLEANUP
# ==============================================================================

print("[0/5] Cleaning previous display/stream processes...")

for name in ["Xvfb", "Xorg", "chromium", "chrome", "ffmpeg"]:
    subprocess.run(f"pkill -9 -x {name}", shell=True)
subprocess.run("pkill -9 -f matchbox-window-manager", shell=True)

time.sleep(2)


# ==============================================================================
# 3. CONFIGURATION
# ==============================================================================

# "both"   = stream live AND save a recording
# "live"   = stream only
# "record" = save a recording only (good for testing, no Facebook/YouTube needed)
MODE = "both"

# Which platform(s) to stream to when MODE is "live" or "both":
#   "both"     = Facebook AND YouTube   (default)
#   "facebook" = Facebook only
#   "youtube"  = YouTube only
STREAM_TO = "both"

# --- RENDERING ----------------------------------------------------------------
# "auto"     = try the GPU paths first, fall back to tuned CPU rendering
# "xorg"     = real NVIDIA X server (GLX)      -> best, if the container allows it
# "vulkan"   = Chromium -> ANGLE -> NVIDIA Vulkan on a virtual screen
# "software" = CPU only (SwiftShader), with speed tricks
RENDER_MODE = "auto"

# Frame rate. Allowed values: 60, 30, 20 or 15 (the browser ticks at 60 Hz, so
# only 60/N values are smooth). GPU paths use FRAMERATE_GPU, the CPU path uses
# FRAMERATE_CPU.
#   60 fps needs the NVENC hardware encoder (T4). If NVENC is not usable the
#   script automatically drops to 30 fps, because CPU x264 cannot keep up.
FRAMERATE_GPU = 60
FRAMERATE_CPU = 30

# Frame rate used automatically when there is no NVENC (CPU x264 encoding).
FRAMERATE_NO_NVENC = 30

# Extra speed tricks used ONLY on the CPU fallback (they slightly soften the look):
SOFTWARE_RENDER_SCALE = 0.75     # render the 3D scene at 75% size, browser upscales
SOFTWARE_DISABLE_AA = True       # turn off anti-aliasing
SOFTWARE_DISABLE_BLUR = True     # remove the blur behind the HUD cards
# ------------------------------------------------------------------------------

# Recording is saved inside Kaggle's working folder (/kaggle/working)
RECORD_PATH = f"/kaggle/working/recording_{int(time.time())}.mkv"

# Upload recordings to your server when finished (record / both modes)
UPLOAD_ENABLED = True
UPLOAD_URL = "https://mukesh-4e2a.onrender.com/upload-video"

# --- FACEBOOK -----------------------------------------------------------------
PAGE_ID = "103517251052640"

# --- YOUTUBE ------------------------------------------------------------------
# "public", "unlisted" or "private"
YOUTUBE_PRIVACY = "public"
YOUTUBE_SCOPES = ["https://www.googleapis.com/auth/youtube"]
# ------------------------------------------------------------------------------

STREAM_TO = STREAM_TO.lower().strip()
if STREAM_TO not in ("both", "facebook", "youtube"):
    raise ValueError('STREAM_TO must be "both", "facebook" or "youtube".')

RENDER_MODE = RENDER_MODE.lower().strip()
if RENDER_MODE not in ("auto", "xorg", "vulkan", "software"):
    raise ValueError('RENDER_MODE must be "auto", "xorg", "vulkan" or "software".')

USE_FACEBOOK = MODE in ("live", "both") and STREAM_TO in ("both", "facebook")
USE_YOUTUBE = MODE in ("live", "both") and STREAM_TO in ("both", "youtube")




def load_secret(name):
    """Read a secret from Kaggle Secrets, then an attached dataset, then env vars."""
    # 1. Kaggle Secrets (works when you run manually in the editor)
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass

    # 2. secrets.json inside an attached private dataset (works for API-triggered runs)
    for path in glob.glob("/kaggle/input/**/secrets.json", recursive=True):
        try:
            with open(path, encoding="utf-8") as f:
                data = json.load(f)
            if name in data:
                value = data[name]
                return value if isinstance(value, str) else json.dumps(value)
        except Exception:
            continue

    # 3. Environment variable
    return os.environ.get(name)


# Load Facebook token from Kaggle Secrets (key name: FB_PAGE_ACCESS_TOKEN)
PAGE_ACCESS_TOKEN = None
if USE_FACEBOOK:
    PAGE_ACCESS_TOKEN = load_secret("FB_PAGE_ACCESS_TOKEN")

    if not PAGE_ACCESS_TOKEN:
        raise RuntimeError(
            "FB_PAGE_ACCESS_TOKEN not found. Add it in Kaggle Secrets "
            "(Add-ons > Secrets) and attach it to this notebook."
        )

# WEBSITE_URL = "https://mukeshpokharel-gorkhapatra.static.hf.space/?date=2026-08-12"
# WEBSITE_URL = "https://mukeshpokharel-graphicalquiz.static.hf.space/"
# WEBSITE_URL = "https://newquiz-main.onrender.com/"
WEBSITE_URL = "https://newquiz-main-hvng.onrender.com/"

# ------------------------------------------------------------------------------
# TITLES / DESCRIPTIONS (built automatically from the date in WEBSITE_URL)
# ------------------------------------------------------------------------------

# If the URL has no ?date=..., use today's date (Nepal time).
# Set to False if you want a title without any date in that case.
USE_TODAY_IF_NO_DATE = True

NEPALI_MONTHS = ["बैशाख", "जेठ", "असार", "श्रावण", "भदौ", "असोज",
                 "कार्तिक", "मंसिर", "पौष", "माघ", "फाल्गुन", "चैत्र"]


def to_nepali_digits(n):
    return str(n).translate(str.maketrans("0123456789", "०१२३४५६७८९"))


def get_date_label(url):
    """Returns a full Nepali date, or None if no date should be used."""
    raw = (parse_qs(urlparse(url).query).get("date") or [None])[0]
    d = None
    if raw:
        try:
            d = datetime.date.fromisoformat(raw.strip())
        except ValueError:
            print(f"[title] Could not read date '{raw}' from URL.")
    if d is None and USE_TODAY_IF_NO_DATE:
        # Kaggle runs in UTC; Nepal is UTC+5:45
        d = (datetime.datetime.now(datetime.timezone.utc)
             + datetime.timedelta(hours=5, minutes=45)).date()
    if d is None:
        return None
    try:
        bs = nepali_datetime.date.from_datetime_date(d)
        return (
            f"{NEPALI_MONTHS[bs.month - 1]} {to_nepali_digits(bs.day)}, "
            f"{to_nepali_digits(bs.year)}"
        )
    except Exception as e:
        print(f"[title] Nepali date conversion failed ({e}); using English date.")
        return d.strftime("%b %d, %Y")


DATE_LABEL = get_date_label(WEBSITE_URL)
DATE_SUFFIX = f" | {DATE_LABEL}" if DATE_LABEL else ""
DATE_LINE = f"📅 मिति: {DATE_LABEL}\n\n" if DATE_LABEL else ""

# ---- FACEBOOK ----
TITLE = f"आजको AI GK Quiz Challenge | नेपाली सामान्य ज्ञान क्विज{DATE_SUFFIX}"
DESCRIPTION = (
    "🧠 आफ्नो GK test गर्न ready हुनुहुन्छ? आजको Nepali AI Quiz Show मा "
    "रोचक questions को answer आफैं सोच्नुहोस्!\n\n"
    "यो show का questions AI-generated हुन्, narration AI voice बाट गरिएको हो, "
    "र quiz को 3D studio पनि AI-assisted रूपमा बनाइएको हो।\n\n"
    f"{DATE_LINE}"
    "⚠️ AI ले कहिलेकाहीँ गलत, अपूर्ण वा पुरानो information दिन सक्छ। त्यसैले "
    "quiz का answers लाई final वा official fact नमान्नुहोस्—महत्त्वपूर्ण जानकारी "
    "विश्वसनीय sources बाट verify गर्नुहोस् र आफ्नो discretion प्रयोग गर्नुहोस्।\n\n"
    "Questions ध्यान दिएर सुन्नुहोस् र आफ्नो answer comment गर्नुहोस्। "
    "Quiz मन परे Like र Follow गर्न नबिर्सनुहोस्!\n\n"
    "#AIQuiz #GKQuiz #सामान्यज्ञान #नेपालीक्विज #NepalGK"
)
# ---- YOUTUBE ----
YT_TITLE = f"AI GK Quiz Challenge 🧠 | नेपाली सामान्य ज्ञान क्विज{DATE_SUFFIX}"[:100]
YT_DESCRIPTION = (
    "Nepali AI GK Quiz Challenge मा welcome! यो quiz show हेरेर आफ्नो सामान्य "
    "ज्ञान test गर्नुहोस् र हरेक question को answer guess गर्नुहोस्।\n\n"
    "Show का questions AI-generated हुन्, narration AI voice मा छ, र 3D quiz "
    "studio पनि AI-assisted रूपमा बनाइएको हो।\n\n"
    f"{DATE_LINE}"
    "⚠️ AI बाट बनेको content मा mistakes हुन सक्छन्—answers गलत, incomplete वा "
    "outdated हुन सक्छन्। त्यसैले यसलाई final वा official information नमान्नुहोस्। "
    "जरुरी facts reliable sources बाट verify गर्नुहोस् र आफ्नै discretion प्रयोग "
    "गर्नुहोस्।\n\n"
    "आफ्नो answer comment गर्नुहोस्, quiz साथीहरूसँग share गर्नुहोस्, र यस्तै "
    "Nepali GK quizzes का लागि channel Subscribe गर्नुहोस्।\n\n"
    "#AIQuiz #GKQuiz #सामान्यज्ञान #नेपालीक्विज #NepalGK"
)
YT_TAGS = [
    "AI quiz", "AI generated quiz", "AI GK quiz", "quiz show",
    "नेपाली क्विज", "सामान्य ज्ञान", "GK", "Nepal GK",
    "current affairs Nepal", "Nepali current affairs", "daily current affairs Nepal",
    "समसामयिक घटना", "लोकसेवा", "लोकसेवा तयारी", "loksewa", "loksewa preparation",
    "general knowledge Nepal", "GK quiz Nepal", "Nepali GK",
    "पुरस्कार", "इतिहास", "Nepal history GK", "awards and prizes GK",
]
YT_CATEGORY_ID = "27"   # 27 = Education

print(f"YouTube title: {YT_TITLE}")

# ------------------------------------------------------------------------------

# 720p-class portrait (4:5). The browser renders at exactly this size and
# FFmpeg captures it as-is.
# Shape of the stream:  "9:16" (tall phone / Reels / Shorts)  or  "4:5"
ASPECT = "9:16"

WIDTH = 720      # layout width the page sees (CSS pixels)
HEIGHT = {"9:16": 1280, "4:5": 900}[ASPECT]     # layout height the page sees

# Output quality. The page keeps the layout above, but is drawn with this many
# pixels per CSS pixel, so the stream is WIDTH*SCALE x HEIGHT*SCALE.
#   9:16:  1.0 = 720x1280    1.5 = 1080x1920 (full HD)
#   4:5 :  1.0 = 720x900     1.5 = 1080x1350
# Use 1.0 if you ever fall back to the CPU "software" render mode, or if the
# [perf] line says TOO SLOW at 60 fps (720x1280 @ 60 is much lighter to capture).
CAPTURE_SCALE = 1.5
OUT_W = int(round(WIDTH * CAPTURE_SCALE / 2)) * 2     # encoders need even numbers
OUT_H = int(round(HEIGHT * CAPTURE_SCALE / 2)) * 2

# Video bitrate (kbps). 60 fps needs more bitrate than 30 fps for the same
# quality. Capped at VIDEO_BITRATE_MAX_K, which is a safe upper range for
# Facebook Live and YouTube at 1080p60. If Facebook shows "bitrate too high"
# or stream health warnings, lower VIDEO_BITRATE_MAX_K to 4500.
VIDEO_BITRATE_MAX_K = 6000
VIDEO_BITRATE_K = (
    min(VIDEO_BITRATE_MAX_K, int(5500 * (OUT_W * OUT_H) / (1080 * 1350)))
    if CAPTURE_SCALE > 1 else 4500
)

FRAMERATE = FRAMERATE_GPU     # final value is chosen after the render mode is known

# --- STREAM START TIMING ------------------------------------------------------
# All the slow setup (Facebook / YouTube API calls) happens BEFORE the website
# is opened. Then FFmpeg starts exactly this many seconds after the page starts
# loading. Counted from the moment the page starts loading.
STREAM_START_DELAY_SECONDS = 10
# ------------------------------------------------------------------------------

# --- FIRESTORE "PLAYING" FLAG (set True by this script, set False by website) --
FIRESTORE_URL = (
    "https://firestore.googleapis.com/v1/projects/loksewa-info/"
    "databases/(default)/documents/app_settings/controls"
)
FIRESTORE_FIELD = "isLiveOngoing"

CONFIRM_DELAY_SECONDS = 7        # wait after setting True before reading back
CONFIRM_RETRY_ATTEMPTS = 3       # how many times to try set + confirm
POLL_INTERVAL_SECONDS = 60       # check the flag every minute
END_GRACE_SECONDS = 3            # small pause after flag goes False before stopping

# Safety net: stop no matter what after this long (in case the website crashes
# before it ever sets the flag to False)
HARD_SAFETY_CAP_MINUTES = 120
# ------------------------------------------------------------------------------


# ==============================================================================
# 3b. FIRESTORE HELPERS
# ==============================================================================

def set_live_status(value: bool) -> bool:
    try:
        r = requests.patch(
            f"{FIRESTORE_URL}?updateMask.fieldPaths={FIRESTORE_FIELD}",
            json={"fields": {FIRESTORE_FIELD: {"booleanValue": value}}},
            timeout=10,
        )
        r.raise_for_status()
        return True
    except Exception as e:
        print(f"[flag] Failed to write value={value}: {e}")
        return False


def get_live_status():
    """Returns True / False, or None if it couldn't be read."""
    try:
        r = requests.get(FIRESTORE_URL, timeout=10)
        r.raise_for_status()
        data = r.json()
        return data.get("fields", {}).get(FIRESTORE_FIELD, {}).get("booleanValue", None)
    except Exception as e:
        print(f"[flag] Failed to read value: {e}")
        return None


def set_and_confirm_flag_true() -> bool:
    """Set the flag True, then read it back to confirm. Retries a few times."""
    global flag_was_set
    for attempt in range(1, CONFIRM_RETRY_ATTEMPTS + 1):
        print(f"[flag] Setting {FIRESTORE_FIELD}=True (attempt {attempt}/{CONFIRM_RETRY_ATTEMPTS})...")
        if set_live_status(True):
            flag_was_set = True
        time.sleep(CONFIRM_DELAY_SECONDS)
        if get_live_status() is True:
            print("[flag] ✅ Confirmed: flag is True.")
            return True
        print("[flag] Could not confirm flag is True yet.")
    return False

def set_firestore_string_field(field_name: str, value: str) -> bool:
    """Updates a specific string field in the Firestore controls document."""
    try:
        r = requests.patch(
            f"{FIRESTORE_URL}?updateMask.fieldPaths={field_name}",
            json={"fields": {field_name: {"stringValue": value}}},
            timeout=10,
        )
        r.raise_for_status()
        print(f"[flag] ✅ Updated Firestore {field_name} = {value}")
        return True
    except Exception as e:
        print(f"[flag] Failed to write {field_name}={value}: {e}")
        return False
# ==============================================================================
# 3c. UPLOAD HELPER
# ==============================================================================

def truncate_text(text, max_len, suffix=""):
    text = (text or "").strip()
    if len(text) <= max_len:
        return text
    return text[:max_len - len(suffix)] + suffix


def upload_video(file_path, description, title=None):
    if not title:
        title = description
    # NOTE: your server may cut titles; change 30 to 100 if it allows longer ones
    title = truncate_text(title, 30, "")

    data = {"description": description, "title": title}

    print(f"\n⬆️ Uploading to {UPLOAD_URL} ...")
    try:
        # httpx streams the file from disk, so large videos don't fill RAM
        with httpx.Client(timeout=httpx.Timeout(900.0, connect=30.0)) as client:
            with open(file_path, "rb") as f:
                files = {"video": ("video.mp4", f, "video/mp4")}
                res = client.post(UPLOAD_URL, files=files, data=data)
        if res.status_code == 200:
            print("✅ Uploaded successfully!")
            return True
        print(f"❌ Upload failed: {res.status_code} | {res.text}")
    except Exception as e:
        print(f"⚠️ Upload error: {e}")
    return False


# ==============================================================================
# 3d. FACEBOOK HELPER
# ==============================================================================

def create_facebook_live():
    """Creates the Facebook live video. Returns the secure stream URL."""
    global live_video_id

    print("\n[3/5] Contacting Facebook Graph API...")

    fb_api_url = f"https://graph.facebook.com/v20.0/{PAGE_ID}/live_videos"

    payload = {
        "title": TITLE,
        "description": DESCRIPTION,
        "status": "LIVE_NOW",
        "access_token": PAGE_ACCESS_TOKEN
    }

    response = requests.post(fb_api_url, data=payload, timeout=60)

    print("Facebook HTTP status:", response.status_code)

    res_data = response.json()

    if "secure_stream_url" not in res_data:
        print("\n❌ FAILED TO CREATE FACEBOOK LIVE STREAM")
        print("Facebook Response:")
        print(res_data)
        raise RuntimeError("Facebook Live broadcast creation failed.")

    fb_stream_url = res_data["secure_stream_url"]
    live_video_id = res_data["id"]

    print("\n✅ Facebook Live Stream Created!")
    print(f"Facebook Live ID: {live_video_id}")

    return fb_stream_url


# ==============================================================================
# 3e. YOUTUBE HELPERS
# ==============================================================================

def build_youtube_service():
    """Builds an authorized YouTube API client from the Kaggle secrets."""
    from google.oauth2.credentials import Credentials
    from google.auth.transport.requests import Request
    from googleapiclient.discovery import build

    token_raw = load_secret("TOKEN_JSON")
    secrets_raw = load_secret("CLIENT_SECRETS_JSON")

    if not token_raw:
        raise RuntimeError("TOKEN_JSON not found in Kaggle Secrets (Add-ons > Secrets).")

    token_info = json.loads(token_raw)

    # Fill in client id/secret from CLIENT_SECRETS_JSON if the token lacks them
    if secrets_raw:
        cs = json.loads(secrets_raw)
        cs = cs.get("installed") or cs.get("web") or cs
        if not token_info.get("client_id"):
            token_info["client_id"] = cs.get("client_id")
        if not token_info.get("client_secret"):
            token_info["client_secret"] = cs.get("client_secret")
        if not token_info.get("token_uri"):
            token_info["token_uri"] = cs.get("token_uri", "https://oauth2.googleapis.com/token")

    scopes = token_info.get("scopes") or YOUTUBE_SCOPES
    creds = Credentials.from_authorized_user_info(token_info, scopes)

    if not creds.valid:
        creds.refresh(Request())

    return build("youtube", "v3", credentials=creds, cache_discovery=False)


def update_youtube_metadata():
    """
    Sets tags, category and language on the live video. (The broadcast API only
    accepts title/description, so tags go through the videos API. For a broadcast,
    the video id is the same as the broadcast id.)
    """
    if yt_service is None or not yt_broadcast_id:
        return
    try:
        yt_service.videos().update(
            part="snippet",
            body={
                "id": yt_broadcast_id,
                "snippet": {
                    "title": YT_TITLE,
                    "description": YT_DESCRIPTION,
                    "tags": YT_TAGS,
                    "categoryId": YT_CATEGORY_ID,
                    "defaultLanguage": "ne",
                    "defaultAudioLanguage": "ne",
                },
            },
        ).execute()
        print("[youtube] ✅ Tags, category and language set.")
    except Exception as e:
        print(f"[youtube] ⚠️ Could not set tags/category (stream is unaffected): {e}")


def create_youtube_live():
    """
    Creates a YouTube broadcast + stream, binds them, and returns the RTMP URL
    (ingestion address + stream key). Auto-start / auto-stop are enabled, so the
    broadcast goes live as soon as FFmpeg starts sending data.
    """
    global yt_service, yt_broadcast_id

    print("\n[3/5] Contacting YouTube Live API...")

    yt_service = build_youtube_service()

    start_time_iso = datetime.datetime.now(datetime.timezone.utc).strftime(
        "%Y-%m-%dT%H:%M:%S.000Z"
    )

    broadcast = yt_service.liveBroadcasts().insert(
        part="snippet,status,contentDetails",
        body={
            "snippet": {
                "title": YT_TITLE,
                "description": YT_DESCRIPTION,
                "scheduledStartTime": start_time_iso,
            },
            "status": {
                "privacyStatus": YOUTUBE_PRIVACY,
                "selfDeclaredMadeForKids": False,
            },
            "contentDetails": {
                "enableAutoStart": True,
                "enableAutoStop": True,
                "enableDvr": True,
                "monitorStream": {"enableMonitorStream": False},
            },
        },
    ).execute()

    yt_broadcast_id = broadcast["id"]
    set_firestore_string_field("youtubeVideoId", yt_broadcast_id)

    stream = yt_service.liveStreams().insert(
        part="snippet,cdn,contentDetails",
        body={
            "snippet": {"title": YT_TITLE},
            "cdn": {
                "frameRate": "variable",
                "ingestionType": "rtmp",
                "resolution": "variable",
            },
            "contentDetails": {"isReusable": False},
        },
    ).execute()

    yt_stream_id = stream["id"]

    yt_service.liveBroadcasts().bind(
        part="id,contentDetails",
        id=yt_broadcast_id,
        streamId=yt_stream_id,
    ).execute()

    # Tags, category, language
    update_youtube_metadata()

    ingestion = stream["cdn"]["ingestionInfo"]
    yt_stream_url = f"{ingestion['ingestionAddress']}/{ingestion['streamName']}"

    print("\n✅ YouTube Live Broadcast Created!")
    print(f"YouTube Broadcast ID: {yt_broadcast_id}")
    print(f"Watch URL: https://www.youtube.com/watch?v={yt_broadcast_id}")

    return yt_stream_url, yt_stream_id


def youtube_wait_until_live(yt_stream_id, timeout_seconds=90):
    """
    Waits for YouTube to receive data and go live. Auto-start normally handles
    this; if it doesn't, we try to transition the broadcast to 'live' ourselves.
    Never raises - problems here are printed as warnings.
    """
    if yt_service is None or not yt_broadcast_id:
        return

    deadline = time.time() + timeout_seconds
    tried_transition = False

    while time.time() < deadline:
        try:
            st = yt_service.liveStreams().list(
                part="status", id=yt_stream_id
            ).execute()
            stream_status = (st.get("items") or [{}])[0].get("status", {}).get("streamStatus")

            bc = yt_service.liveBroadcasts().list(
                part="status", id=yt_broadcast_id
            ).execute()
            life = (bc.get("items") or [{}])[0].get("status", {}).get("lifeCycleStatus")

            print(f"[youtube] stream={stream_status} broadcast={life}")

            if life == "live":
                print("[youtube] ✅ Broadcast is LIVE.")
                return

            if stream_status == "active" and life in ("ready", "testing") and not tried_transition:
                tried_transition = True
                try:
                    yt_service.liveBroadcasts().transition(
                        broadcastStatus="live",
                        id=yt_broadcast_id,
                        part="status",
                    ).execute()
                    print("[youtube] Requested transition to LIVE.")
                except Exception as e:
                    print(f"[youtube] Transition request failed (auto-start may still work): {e}")
        except Exception as e:
            print(f"[youtube] Status check error: {e}")

        time.sleep(5)

    print("[youtube] ⚠️ Did not confirm LIVE within the timeout. "
          "Check YouTube Studio; it may still go live shortly.")


def end_youtube_live():
    """Ends the YouTube broadcast (transition to 'complete')."""
    if yt_service is None or not yt_broadcast_id:
        return
    try:
        yt_service.liveBroadcasts().transition(
            broadcastStatus="complete",
            id=yt_broadcast_id,
            part="status",
        ).execute()
        print("YouTube Live ended.")
    except Exception as e:
        print(f"Could not explicitly end YouTube Live (it may auto-end): {e}")


# ==============================================================================
# 4. GPU HELPERS  (environment, NVENC test, display servers, browser flags)
# ==============================================================================

def gpu_env(use_nv=False, display=None):
    """Environment for a child process. With use_nv=True the NVIDIA graphics
    libraries (OpenGL / Vulkan / EGL / NVENC) are made visible to it."""
    e = dict(os.environ)
    if use_nv and NV_OK:
        e["LD_LIBRARY_PATH"] = NV_LIBS + (":" + e["LD_LIBRARY_PATH"] if e.get("LD_LIBRARY_PATH") else "")
        e["VK_ICD_FILENAMES"] = os.path.join(NV_LIBS, "nvidia_icd.json")
        e["__EGL_VENDOR_LIBRARY_FILENAMES"] = os.path.join(NV_LIBS, "10_nvidia.json")
    if display:
        e["DISPLAY"] = display
    return e


def pick_nvenc():
    """Returns working NVENC encoder arguments, or None if NVENC is unusable
    (for example on a P100, which has no hardware video encoder)."""
    if not HAS_NVIDIA:
        return None
    candidates = [
        ["-c:v", "h264_nvenc", "-preset", "p4", "-tune", "ll"],
        ["-c:v", "h264_nvenc", "-preset", "llhq"],
        ["-c:v", "h264_nvenc"],
    ]
    for c in candidates:
        try:
            r = subprocess.run(
                ["ffmpeg", "-hide_banner", "-loglevel", "error",
                 "-f", "lavfi", "-i", "testsrc=size=640x480:rate=25",
                 "-frames:v", "20", "-pix_fmt", "yuv420p", *c, "-f", "null", "-"],
                env=gpu_env(use_nv=True), capture_output=True, text=True, timeout=60
            )
        except Exception:
            continue
        if r.returncode == 0:
            return c
    return None


def _force_symlink(src, dst):
    try:
        if os.path.lexists(dst):
            os.remove(dst)
        os.symlink(src, dst)
    except Exception:
        pass


def _x_socket(disp):
    return f"/tmp/.X11-unix/X{disp.lstrip(':')}"


def _wait_x(disp, proc, timeout=12):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            return False
        if os.path.exists(_x_socket(disp)):
            time.sleep(1)
            return True
        time.sleep(0.3)
    return False


def _pci_to_xorg(raw):
    """'00000000:00:04.0' -> 'PCI:0:4:0'"""
    m = re.match(r"[0-9A-Fa-f]+:([0-9A-Fa-f]+):([0-9A-Fa-f]+)\.([0-9A-Fa-f]+)", raw or "")
    if not m:
        return None
    return f"PCI:{int(m.group(1), 16)}:{int(m.group(2), 16)}:{int(m.group(3), 16)}"


def _start_xorg_nvidia(disp):
    """A real X server driven by the NVIDIA GPU (no monitor needed)."""
    if not NV_OK:
        raise RuntimeError("NVIDIA graphics libraries not available")
    xorg = "/usr/lib/xorg/Xorg" if os.path.exists("/usr/lib/xorg/Xorg") else shutil.which("Xorg")
    if not xorg:
        raise RuntimeError("Xorg is not installed")
    drv = os.path.join(NV_DIR, "nvidia_drv.so")
    glx = glob.glob(os.path.join(NV_DIR, "libglxserver_nvidia.so.*"))
    if not os.path.exists(drv) or not glx:
        raise RuntimeError("NVIDIA X driver files are missing from the package")

    root = "/tmp/nvx"
    os.makedirs(f"{root}/drivers", exist_ok=True)
    os.makedirs(f"{root}/extensions", exist_ok=True)
    _force_symlink(drv, f"{root}/drivers/nvidia_drv.so")
    _force_symlink(glx[0], f"{root}/extensions/libglx.so")
    _force_symlink(glx[0], f"{root}/extensions/{os.path.basename(glx[0])}")

    busid = _pci_to_xorg(NV_BUSID)
    cfg = f"""
Section "ServerFlags"
    Option "AutoAddDevices" "false"
    Option "AutoEnableDevices" "false"
    Option "AutoAddGPU" "false"
EndSection
Section "Files"
    ModulePath "{root}"
    ModulePath "/usr/lib/xorg/modules"
EndSection
Section "Device"
    Identifier "gpu0"
    Driver "nvidia"
{f'    BusID "{busid}"' if busid else ''}
    Option "AllowEmptyInitialConfiguration" "True"
    Option "UseDisplayDevice" "None"
EndSection
Section "Screen"
    Identifier "screen0"
    Device "gpu0"
    DefaultDepth 24
    SubSection "Display"
        Depth 24
        Virtual {OUT_W} {OUT_H}
    EndSubSection
EndSection
Section "ServerLayout"
    Identifier "layout0"
    Screen 0 "screen0"
EndSection
"""
    cfg_path = "/tmp/xorg_nvidia.conf"
    with open(cfg_path, "w") as fh:
        fh.write(cfg)
    log_path = "/tmp/xorg_nvidia.log"
    if os.path.exists(log_path):
        os.remove(log_path)

    proc = subprocess.Popen(
        [xorg, disp, "-config", cfg_path, "-logfile", log_path,
         "-noreset", "-novtswitch", "-sharevts", "-ac"],
        env=gpu_env(use_nv=True), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    if not _wait_x(disp, proc):
        try:
            with open(log_path) as fh:
                errs = [l.rstrip() for l in fh if "(EE)" in l][-8:]
            print("   Xorg log (errors):")
            for l in errs:
                print("   ", l)
        except Exception:
            pass
        try:
            proc.terminate()
        except Exception:
            pass
        raise RuntimeError("Xorg with the NVIDIA driver did not start")
    return proc


def start_display(kind, disp):
    """Start the virtual screen for a render mode. Returns the process."""
    if kind == "xorg":
        return _start_xorg_nvidia(disp)
    proc = subprocess.Popen(
        ["Xvfb", disp, "-screen", "0", f"{OUT_W}x{OUT_H}x24", "-ac", "-noreset"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    if not _wait_x(disp, proc):
        try:
            proc.terminate()
        except Exception:
            pass
        raise RuntimeError("Xvfb did not start")
    return proc


def _stop(proc):
    if proc is None:
        return
    try:
        proc.terminate()
        proc.wait(timeout=5)
    except Exception:
        try:
            proc.kill()
        except Exception:
            pass


# Chromium arguments shared by every mode (no GPU-related ones here)
BASE_CHROME_ARGS = [
    f"--window-size={OUT_W},{OUT_H}",
    "--window-position=0,0",
    "--kiosk",
    "--start-fullscreen",
    "--no-first-run",
    "--no-default-browser-check",
    "--disable-infobars",
    "--no-sandbox",
    "--disable-setuid-sandbox",
    "--disable-dev-shm-usage",
    "--autoplay-policy=no-user-gesture-required",
    "--use-fake-ui-for-media-stream",
    "--use-fake-device-for-media-stream",
    f"--force-device-scale-factor={CAPTURE_SCALE}",
    "--overscroll-history-navigation=0",
    "--noerrdialogs",
    "--disable-session-crashed-bubble",
    "--disable-translate",
    "--disable-notifications",
    # anti-throttling
    "--disable-background-timer-throttling",
    "--disable-renderer-backgrounding",
    "--disable-backgrounding-occluded-windows",
    "--disable-features=TranslateUI,CalculateNativeWinOcclusion",
    "--disable-ipc-flooding-protection",
]

MODE_CHROME_FLAGS = {
    # Real NVIDIA X server: WebGL and page compositing both on the GPU
    "xorg": [
        "--use-gl=angle", "--use-angle=gl",
        "--ignore-gpu-blocklist", "--enable-gpu-rasterization",
        "--disable-gpu-sandbox",
    ],
    # Virtual screen, WebGL on the GPU through Vulkan (no X-server GL needed)
    "vulkan": [
        "--use-gl=angle", "--use-angle=vulkan",
        "--enable-features=Vulkan,VulkanFromANGLE,DefaultANGLEVulkan",
        "--disable-vulkan-surface",
        "--ignore-gpu-blocklist", "--enable-gpu-rasterization",
        "--disable-gpu-sandbox",
    ],
    # CPU only: SwiftShader for WebGL, software page compositing
    "software": [
        "--use-gl=angle", "--use-angle=swiftshader", "--enable-unsafe-swiftshader",
        "--ignore-gpu-blocklist", "--disable-gpu-compositing",
        "--disable-smooth-scrolling", "--disable-lcd-text",
    ],
}

# Injected into the page before its own scripts run (your page file is NOT edited):
#  * limits requestAnimationFrame to the capture frame rate (no wasted frames)
#  * at 60 fps the limiter lets every browser tick through (it only counts frames)
#  * asks for the high-performance GPU
#  * CPU mode only: no anti-aliasing, lower render scale, no blur behind HUD cards
INIT_JS = r"""
(() => {
  const OPT = __OPT__;
  const realRAF = window.requestAnimationFrame.bind(window);
  const gap = 1000 / OPT.fps - 4;
  let lastRun = -1e9, curTs = -1;
  window.__frames = 0;
  window.requestAnimationFrame = function (cb) {
    const w = function (ts) {
      if (ts === curTs) { cb(ts); return; }
      if (OPT.fps < 60 && ts - lastRun < gap) { realRAF(w); return; }
      lastRun = ts; curTs = ts; window.__frames++;
      cb(ts);
    };
    return realRAF(w);
  };

  const gc = HTMLCanvasElement.prototype.getContext;
  HTMLCanvasElement.prototype.getContext = function (type, attrs) {
    if (typeof type === 'string' && type.indexOf('webgl') === 0) {
      attrs = Object.assign({}, attrs || {}, {
        powerPreference: 'high-performance',
        failIfMajorPerformanceCaveat: false
      });
      if (OPT.noAA) attrs.antialias = false;
    }
    return gc.call(this, type, attrs);
  };

  if (OPT.dpr) {
    try { Object.defineProperty(window, 'devicePixelRatio', { get: () => OPT.dpr, configurable: true }); } catch (e) {}
  }

  if (OPT.noBlur) {
    const add = () => {
      const s = document.createElement('style');
      s.textContent = '.card{backdrop-filter:none!important;-webkit-backdrop-filter:none!important}';
      (document.head || document.documentElement).appendChild(s);
    };
    if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', add); else add();
  }
})();
"""

PROBE_JS = """() => {
  try {
    const c = document.createElement('canvas');
    const gl = c.getContext('webgl2') || c.getContext('webgl');
    if (!gl) return 'NO_WEBGL';
    const e = gl.getExtension('WEBGL_debug_renderer_info');
    return e ? gl.getParameter(e.UNMASKED_RENDERER_WEBGL) : 'unknown';
  } catch (err) { return 'ERR ' + err; }
}"""


def looks_like_gpu(renderer):
    s = (renderer or "").lower()
    if any(k in s for k in ("swiftshader", "llvmpipe", "software", "no_webgl", "err ")):
        return False
    return any(k in s for k in ("nvidia", "tesla", "geforce", "quadro", " t4", "p100"))


def snap_fps(f):
    return min((60, 30, 20, 15), key=lambda x: abs(x - f))


def build_init_script(rm, fps):
    sw = rm == "software"
    opt = {
        "fps": fps,
        "noAA": bool(sw and SOFTWARE_DISABLE_AA),
        "dpr": SOFTWARE_RENDER_SCALE if sw and SOFTWARE_RENDER_SCALE and SOFTWARE_RENDER_SCALE < 1 else None,
        "noBlur": bool(sw and SOFTWARE_DISABLE_BLUR),
    }
    return INIT_JS.replace("__OPT__", json.dumps(opt))


def nvidia_stats():
    if not HAS_NVIDIA:
        return ""
    rc, o = _out("nvidia-smi --query-gpu=utilization.gpu,utilization.memory,encoder.stats.sessionCount "
                 "--format=csv,noheader")
    if rc != 0 or not o:
        return ""
    return " | GPU util/mem/enc-sessions: " + o.splitlines()[0]


NVENC_ARGS = pick_nvenc()
print(f"\nNVIDIA GPU: {NV_NAME if HAS_NVIDIA else 'none'}")
print(f"NVENC hardware encoder usable: {bool(NVENC_ARGS)}")
print(f"GPU graphics libraries ready (for WebGL): {NV_OK}")
print(f"Mode: {MODE}")
if MODE in ("live", "both"):
    print(f"Stream to: {STREAM_TO}")

DISPLAY_NUM = None   # chosen when a render mode succeeds


# ==============================================================================
# 5. AUDIO  (virtual sink; the screen is started later, per render mode)
# ==============================================================================

print("\nStarting PulseAudio...")
pulseaudio_proc = subprocess.Popen(["pulseaudio", "--start", "--exit-idle-time=-1"])
time.sleep(2)

subprocess.run("pactl unload-module module-null-sink 2>/dev/null || true", shell=True)

print("Creating VirtualSink...")
subprocess.run(
    "pactl load-module module-null-sink "
    "sink_name=VirtualSink "
    "sink_properties=device.description=VirtualSink",
    shell=True,
    check=True
)
subprocess.run("pactl set-default-sink VirtualSink", shell=True, check=True)


# ==============================================================================
# 6. MAIN
# ==============================================================================

async def launch_browser(p, rm, disp):
    flags = BASE_CHROME_ARGS + MODE_CHROME_FLAGS[rm] + [f"--window-size={OUT_W},{OUT_H}"]
    ctx = await p.chromium.launch_persistent_context(
        user_data_dir=tempfile.mkdtemp(prefix="quiz_live_"),
        headless=False,
        no_viewport=True,
        permissions=["microphone", "camera"],
        ignore_default_args=["--enable-automation"],
        args=flags,
        env=gpu_env(use_nv=(rm in ("xorg", "vulkan")), display=disp),
    )
    await ctx.add_init_script(script=build_init_script(rm, FRAMERATE))
    return ctx


async def read_frames(page):
    try:
        return await page.evaluate("window.__frames || 0")
    except Exception:
        return None


async def main():

    global ffmpeg_proc, live_video_id, display_proc, wm_proc, DISPLAY_NUM, FRAMERATE

    async with async_playwright() as p:

        # ----------------------------------------------------------------------
        # Pick the best working render mode (GPU first, CPU as the last resort)
        # ----------------------------------------------------------------------
        if RENDER_MODE != "auto":
            order = [RENDER_MODE]
        else:
            order = (["xorg", "vulkan"] if NV_OK else []) + ["software"]

        context = None
        page = None
        chosen = None

        for rm in order:
            print(f"\n[render] Trying mode: {rm}")
            FRAMERATE = snap_fps(FRAMERATE_CPU if rm == "software" else FRAMERATE_GPU)
            # Without the NVENC hardware encoder, CPU x264 cannot encode 60 fps
            # in real time, so fall back to a lower frame rate.
            if not NVENC_ARGS and FRAMERATE > FRAMERATE_NO_NVENC:
                print(f"[render] ⚠️ NVENC not available - lowering frame rate from "
                      f"{FRAMERATE} to {FRAMERATE_NO_NVENC} fps.")
                FRAMERATE = snap_fps(FRAMERATE_NO_NVENC)
            if rm == "software" and CAPTURE_SCALE > 1:
                print("[render] ⚠️ CPU mode at high resolution will be slow; set CAPTURE_SCALE = 1.0 for it.")
            disp = f":{random.randint(50, 98)}"
            context = None
            try:
                display_proc = start_display(rm, disp)
                os.environ["DISPLAY"] = disp
                wm_proc = subprocess.Popen(
                    ["matchbox-window-manager", "-use_titlebar", "no"],
                    env=gpu_env(display=disp),
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
                )
                time.sleep(1)

                print("Starting Chromium...")
                context = await launch_browser(p, rm, disp)
                page = context.pages[0] if context.pages else await context.new_page()

                renderer = await page.evaluate(PROBE_JS)
                print(f"[render] WebGL renderer reported by Chromium: {renderer}")
                vp = await page.evaluate("[innerWidth, innerHeight, devicePixelRatio]")
                print(f"[render] Page viewport: {vp[0]}x{vp[1]} CSS px at pixel ratio {vp[2]} "
                      f"-> {round(vp[0] * vp[2])}x{round(vp[1] * vp[2])} real pixels "
                      f"(capture is {OUT_W}x{OUT_H})")

                if rm != "software" and not looks_like_gpu(renderer):
                    raise RuntimeError("Chromium is not using the GPU in this mode")

                chosen = rm
                DISPLAY_NUM = disp
                break

            except Exception as e:
                print(f"[render] ❌ Mode '{rm}' failed: {e}")
                try:
                    if context is not None:
                        await context.close()
                except Exception:
                    pass
                _stop(wm_proc)
                _stop(display_proc)
                wm_proc = None
                display_proc = None
                time.sleep(1)

        if chosen is None:
            raise RuntimeError("No render mode worked (not even software).")

        encoder_name = "NVENC (GPU)" if NVENC_ARGS else "libx264 (CPU)"
        print("\n" + "-" * 70)
        print(f"[render] ✅ Using render mode: {chosen.upper()}"
              f"{'  (WebGL on the GPU)' if chosen != 'software' else '  (CPU only)'}")
        print(f"[render] Frame rate: {FRAMERATE} fps | Encoder: {encoder_name}")
        print("-" * 70)

        # Keep the browser on a blank page while the slow setup (Facebook /
        # YouTube API calls) happens. The website is only opened afterwards,
        # and nothing is streamed until then.
        print("\n[2/5] Browser ready (blank page). Preparing live broadcasts first...")

        print("Making Chromium fullscreen...")
        subprocess.run('wmctrl -r ":ACTIVE:" -b add,fullscreen', shell=True)
        await asyncio.sleep(2)

        # ======================================================================
        # 3. CREATE LIVE BROADCASTS (Facebook and/or YouTube)
        # ======================================================================

        fb_stream_url = None
        yt_stream_url = None
        yt_stream_id = None

        if MODE in ("live", "both"):

            errors = []

            if USE_FACEBOOK:
                try:
                    fb_stream_url = create_facebook_live()
                except Exception as e:
                    errors.append(f"Facebook: {e}")
                    print(f"⚠️ Facebook setup failed: {e}")

            if USE_YOUTUBE:
                try:
                    yt_stream_url, yt_stream_id = create_youtube_live()
                except Exception as e:
                    errors.append(f"YouTube: {e}")
                    print(f"⚠️ YouTube setup failed: {e}")

            if not fb_stream_url and not yt_stream_url:
                raise RuntimeError(
                    "No live destination could be created. " + " | ".join(errors)
                )

            if errors:
                print("\n⚠️ Continuing with the destination(s) that worked.")

        else:
            print("\n[3/5] Record-only mode — skipping Facebook/YouTube.")

        # ======================================================================
        # 4. START FFMPEG
        # ======================================================================

        print("\n[4/5] Preparing FFmpeg command...")

        # Keyframe every 2 seconds (platforms require 2 s or less):
        # 60 fps -> -g 120, 30 fps -> -g 60
        if NVENC_ARGS:
            video_encode_args = [
                *NVENC_ARGS,
                "-b:v", f"{VIDEO_BITRATE_K}k",
                "-maxrate", f"{int(VIDEO_BITRATE_K * 1.15)}k",
                "-bufsize", f"{VIDEO_BITRATE_K * 2}k",
                "-pix_fmt", "yuv420p",
                "-g", str(FRAMERATE * 2),
                "-bf", "0",
            ]
        else:
            video_encode_args = [
                "-c:v", "libx264",
                "-preset", "ultrafast",
                "-tune", "zerolatency",
                "-b:v", f"{min(VIDEO_BITRATE_K, 3000)}k",
                "-maxrate", f"{min(int(VIDEO_BITRATE_K * 1.15), 3400)}k",
                "-bufsize", f"{min(VIDEO_BITRATE_K, 3000) * 2}k",
                "-pix_fmt", "yuv420p",
                "-g", str(FRAMERATE * 2),
                "-threads", "2",
            ]

        ffmpeg_cmd = [
            "ffmpeg", "-y",

            # Video input
            "-thread_queue_size", "1024",
            "-f", "x11grab",
            "-use_wallclock_as_timestamps", "1",
            "-video_size", f"{OUT_W}x{OUT_H}",
            "-framerate", str(FRAMERATE),
            "-draw_mouse", "0",
            "-i", DISPLAY_NUM,

            # Audio input
            "-thread_queue_size", "512",
            "-f", "pulse",
            "-i", "VirtualSink.monitor",

            "-map", "0:v",
            "-map", "1:a",

            # Force a steady output frame rate (duplicates/drops frames if the
            # grab is slightly uneven, so the stream stays a clean constant 60)
            "-r", str(FRAMERATE),

            *video_encode_args,

            "-c:a", "aac",
            "-b:a", "128k",
            "-ar", "44100",
            "-af", "aresample=async=1000",

            "-flags", "+global_header",
        ]

        # Build the list of output destinations as (format, target)
        outputs = []
        if fb_stream_url:
            outputs.append(("flv", fb_stream_url))
        if yt_stream_url:
            outputs.append(("flv", yt_stream_url))
        if MODE in ("both", "record"):
            outputs.append(("matroska", RECORD_PATH))

        if len(outputs) == 1:
            # Single destination
            fmt, target = outputs[0]
            ffmpeg_cmd += ["-f", fmt, target]
        else:
            # Multiple destinations: one encode, many outputs via tee.
            # onfail=ignore: if one platform drops, the others keep going.
            tee_parts = []
            for fmt, target in outputs:
                if fmt == "flv":
                    tee_parts.append(f"[f=flv:onfail=ignore]{target}")
                else:
                    tee_parts.append(f"[f=matroska]{target}")
            ffmpeg_cmd += ["-f", "tee", "|".join(tee_parts)]

        # ======================================================================
        # 4a. OPEN THE WEBSITE, THEN START STREAMING EXACTLY N SECONDS LATER
        # ======================================================================

        print("\n[2/5] Opening website...")
        print(f"Website URL: {WEBSITE_URL}")

        nav_start = time.time()
        await page.goto(WEBSITE_URL, wait_until="domcontentloaded", timeout=120000)

        try:
            await page.evaluate(
                "document.activeElement && document.activeElement.blur();"
            )
        except Exception:
            pass

        print(f"✅ Website opened ({time.time() - nav_start:.1f}s after navigation started).")

        wait_left = STREAM_START_DELAY_SECONDS - (time.time() - nav_start)
        if wait_left <= 0:
            print(
                f"⚠️ The page took longer than {STREAM_START_DELAY_SECONDS}s to open; "
                "starting the stream immediately."
            )
        while wait_left > 0:
            print(f"   🔵 Stream starts in {wait_left:.0f}s...")
            await asyncio.sleep(min(1, wait_left))
            wait_left = STREAM_START_DELAY_SECONDS - (time.time() - nav_start)

        print("\n[4/5] Starting FFmpeg now...")
        print("FFmpeg started (command hidden to protect the stream keys).")

        ffmpeg_proc = subprocess.Popen(
            ffmpeg_cmd,
            stdout=subprocess.DEVNULL,
            stderr=None,
            env=gpu_env(use_nv=bool(NVENC_ARGS)),
        )

        await asyncio.sleep(5)

        if ffmpeg_proc.poll() is not None:
            print("\n❌ FFmpeg EXITED IMMEDIATELY!")
            print("Exit code:", ffmpeg_proc.returncode)
            raise RuntimeError("FFmpeg failed to start. Check the FFmpeg error printed above.")

        print("✅ FFmpeg is running.")

        # YouTube: check in the background that it goes live
        yt_wait_task = None
        if yt_stream_url:
            print("\n[youtube] Checking in the background that YouTube goes live...")
            yt_wait_task = asyncio.get_event_loop().run_in_executor(
                None, youtube_wait_until_live, yt_stream_id
            )

        # ======================================================================
        # 4b. SET + CONFIRM THE "PLAYING" FLAG
        # ======================================================================

        print("\n[flag] Telling the website that recording/live has started...")
        flag_confirmed = set_and_confirm_flag_true()

        if not flag_confirmed:
            print(
                "[flag] ⚠️ Could not confirm the flag. Auto-stop may not work; "
                f"the {HARD_SAFETY_CAP_MINUTES}-minute safety cap will still apply."
            )

        if yt_wait_task is not None:
            await yt_wait_task

        # ======================================================================
        # 5. RUNNING
        # ======================================================================

        print("\n" + "=" * 70)
        print("🔴 BROADCAST / RECORDING IS RUNNING!")
        print("=" * 70)

        print(f"Mode: {MODE}")
        if MODE in ("live", "both"):
            dests = []
            if fb_stream_url:
                dests.append("Facebook")
            if yt_stream_url:
                dests.append("YouTube")
            print(f"Streaming to: {', '.join(dests)}")
        print(f"Title: {TITLE}")
        print(f"Stream: {OUT_W}x{OUT_H} @ {FRAMERATE}fps | {VIDEO_BITRATE_K} kbps")
        print(f"Render: {chosen} | Encoder: {encoder_name}")
        print(f"Website: {WEBSITE_URL}")
        if live_video_id:
            print(f"Facebook Live ID: {live_video_id}")
        if yt_broadcast_id:
            print(f"YouTube: https://www.youtube.com/watch?v={yt_broadcast_id}")
        if MODE in ("record", "both"):
            print(f"Recording to: {RECORD_PATH}")

        print(f"\nChecking '{FIRESTORE_FIELD}' every {POLL_INTERVAL_SECONDS}s.")
        print(f"Will stop when it becomes False (safety cap: {HARD_SAFETY_CAP_MINUTES} min).")
        print("You can also interrupt this cell manually at any time.")

        max_seconds = HARD_SAFETY_CAP_MINUTES * 60
        start_time = time.time()
        next_poll = start_time + POLL_INTERVAL_SECONDS

        last_frames = await read_frames(page)
        last_frames_t = time.time()

        while True:

            if ffmpeg_proc.poll() is not None:
                print("\n❌ FFmpeg stopped unexpectedly!")
                print("FFmpeg exit code:", ffmpeg_proc.returncode)
                break

            now = time.time()

            if now - start_time >= max_seconds:
                print(f"\n⏱️ Reached {HARD_SAFETY_CAP_MINUTES} minute safety cap — stopping.")
                break

            if now >= next_poll:
                next_poll = now + POLL_INTERVAL_SECONDS
                status = get_live_status()
                elapsed_min = (now - start_time) / 60
                print(f"[flag] t={elapsed_min:.1f} min → {FIRESTORE_FIELD} = {status}")

                # Performance report: how many frames the page really draws
                fr = await read_frames(page)
                if fr is not None and last_frames is not None and now > last_frames_t:
                    fps_now = (fr - last_frames) / (now - last_frames_t)
                    verdict = "OK" if fps_now >= FRAMERATE * 0.9 else "TOO SLOW"
                    print(f"[perf] page renders {fps_now:.1f} fps (target {FRAMERATE}) → {verdict}"
                          f"{nvidia_stats()}")
                    last_frames, last_frames_t = fr, now

                # Only an explicit False ends the run. None (couldn't read)
                # is treated as "unknown, keep going".
                if status is False:
                    print("\n✅ Website finished (flag is False) — stopping now.")
                    await asyncio.sleep(END_GRACE_SECONDS)
                    break

            await asyncio.sleep(5)


# ==============================================================================
# 7. RUN
# ==============================================================================

try:
    await main()

except (KeyboardInterrupt, asyncio.CancelledError):
    print("\nStopping (manual interrupt)...")

except Exception as e:
    print(f"\n❌ Error: {e}")

finally:

    print("\nCleaning up...")

    # Tell the website/other programs that we're no longer running.
    # This covers normal finish, manual stop, and errors.
    if flag_was_set:
        if set_live_status(False):
            print("[flag] Set flag to False.")

    # Stop FFmpeg gracefully (SIGINT lets it finalize the recording file)
    if ffmpeg_proc is not None:
        try:
            if ffmpeg_proc.poll() is None:
                ffmpeg_proc.send_signal(signal.SIGINT)
            ffmpeg_proc.wait(timeout=20)
        except Exception:
            try:
                ffmpeg_proc.kill()
            except Exception:
                pass

    # Explicitly end the Facebook live so it doesn't linger
    if live_video_id and PAGE_ACCESS_TOKEN:
        try:
            requests.post(
                f"https://graph.facebook.com/v20.0/{live_video_id}",
                data={"end_live_video": "true", "access_token": PAGE_ACCESS_TOKEN},
                timeout=20,
            )
            print("Facebook Live ended.")
        except Exception as e:
            print(f"Could not explicitly end Facebook Live: {e}")

    # Explicitly end the YouTube live
    end_youtube_live()

    for proc in (wm_proc, display_proc, pulseaudio_proc):
        if proc is not None:
            try:
                proc.terminate()
            except Exception:
                pass

    # Convert recording to MP4 (no re-encode, takes seconds), then upload
    if MODE in ("record", "both") and os.path.exists(RECORD_PATH):
        mp4_path = RECORD_PATH.replace(".mkv", ".mp4")
        print("\nConverting recording to MP4...")
        subprocess.run(
            ["ffmpeg", "-y", "-i", RECORD_PATH, "-c", "copy",
             "-movflags", "+faststart", mp4_path],
            check=False
        )

        if os.path.exists(mp4_path):
            size_mb = os.path.getsize(mp4_path) / (1024 * 1024)
            print(f"✅ Saved: {mp4_path} ({size_mb:.1f} MB)")

            if UPLOAD_ENABLED:
                upload_video(mp4_path, DESCRIPTION, TITLE)
            else:
                print("Upload disabled (UPLOAD_ENABLED = False).")

            print("Recording is also in the Output panel (right side) under /kaggle/working.")
        else:
            print(f"MP4 conversion failed. The raw recording is still at: {RECORD_PATH}")

    print("Stopped cleanly.")